# Rudimentary analysis of a video using Qwen 2.5VL 7B model
Attempted pruning but cuda runs out of memory (!try)


In [ ]:
#installing packages
!pip install torch torchvision torchaudio --quiet
!pip install transformers datasets accelerate --quiet
!pip install huggingface_hub tqdm matplotlib numpy --quiet
!pip install opencv-python --quiet
!pip install git+https://github.com/LLaVA-VL/LLaVA-NeXT.git --quiet
!pip install decord --quiet


In [ ]:
import torch
import cv2
import numpy as np
import os
import warnings
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from PIL import Image
import torch.nn.utils.prune as prune

warnings.filterwarnings("ignore")


In [ ]:
# Load official Qwen2.5-Omni model from Hugging Face
model_name = "Qwen/Qwen2.5-Omni-7B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name}...")
print(f"Using device: {device}")

# Load the text-only model for efficiency
model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

processor = Qwen2_5OmniProcessor.from_pretrained(model_name)

print("Model and processor loaded successfully!")
print(f"Model device: {model.device}")


In [ ]:
def prune_model_random_unstructured(model, amount=0.9):
    """
    Randomly prunes `amount` of weights in all prunable layers of the model.
    Only weights (not biases) are pruned.

    Args:
        model: The model to prune
        amount: Fraction of weights to prune (0.0 to 1.0)

    Returns:
        Pruned model
    """
    pruned_layers = 0
    total_layers = 0

    for name, module in model.named_modules():
        total_layers += 1
        # Prune only Linear and Conv2d layers
        if isinstance(module, torch.nn.Linear) or isinstance(module, torch.nn.Conv2d):
            prune.random_unstructured(module, name="weight", amount=amount)
            pruned_layers += 1

    print(f"Pruned {pruned_layers} layers out of {total_layers} total layers")
    return model

# Apply pruning (optional - uncomment to use)
# print("Applying pruning...")
# model = prune_model_random_unstructured(model, amount=0.9)
# print("Pruning completed!")


In [ ]:
def load_video_frames(video_path, max_frames=8, target_size=(224, 224)):
    """
    Load video frames using OpenCV

    Args:
        video_path: Path to video file
        max_frames: Maximum number of frames to extract
        target_size: Target size for frames (width, height)

    Returns:
        List of PIL Images
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")

    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / fps if fps > 0 else 0

    if frame_count == 0:
        raise ValueError(f"No frames found in video: {video_path}")

    # Calculate step size to get evenly distributed frames
    step = max(1, frame_count // max_frames)

    print(f"Video info: {frame_count} frames, {fps:.2f} FPS, {duration:.2f}s duration")
    print(f"Extracting every {step} frames to get {max_frames} frames")

    for i in range(0, frame_count, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()

        if ret:
            # Convert BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Resize frame
            frame = cv2.resize(frame, target_size)
            # Convert to PIL Image
            pil_image = Image.fromarray(frame)
            frames.append(pil_image)

        if len(frames) >= max_frames:
            break

    cap.release()

    if not frames:
        raise ValueError(f"No frames could be extracted from video: {video_path}")

    return frames

def test_video_loading(video_path):
    """Test function to verify video loading works"""
    try:
        frames = load_video_frames(video_path, max_frames=3)
        print(f"✓ Successfully loaded {len(frames)} frames")
        print(f"✓ Frame dimensions: {frames[0].size}")
        return True
    except Exception as e:
        print(f"✗ Video loading failed: {e}")
        return False


In [ ]:
def inference_single_frame(model, processor, video_path, prompt, sys_prompt, max_frames=8):
    """
    Inference function using single representative frame from video

    Args:
        model: The loaded model
        processor: The processor
        video_path: Path to video file
        prompt: User prompt
        sys_prompt: System prompt
        max_frames: Maximum frames to extract from video

    Returns:
        Generated text response
    """
    try:
        # Load video frames using OpenCV
        frames = load_video_frames(video_path, max_frames=max_frames)
        print(f"Extracted {len(frames)} frames from video")

        # Use first frame as representative image
        representative_frame = frames[0]

        # Create conversation with image
        conversations = [
            {
                "role": "system",
                "content": [{"type": "text", "text": sys_prompt}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": representative_frame},
                    {"type": "text", "text": f"{prompt} (Note: This is a frame from a video with {len(frames)} total frames)"},
                ],
            },
        ]

        # Process the conversation
        inputs = processor.apply_chat_template(
            conversations,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )

        # Move inputs to model device
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generate response
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                pad_token_id=processor.tokenizer.eos_token_id
            )

            # Extract only the new tokens (response)
            prompt_length = inputs["input_ids"].size(1)
            completion_ids = generated_ids[:, prompt_length:]

            # Decode the response
            text = processor.batch_decode(
                completion_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )

        return text[0] if text else "No response generated"

    except Exception as e:
        print(f"Error during inference: {e}")
        return f"Error processing video: {str(e)}"


In [ ]:
def inference_multi_frame(model, processor, video_path, prompt, sys_prompt, max_frames=4):
    """
    Enhanced inference function that processes multiple frames from video

    Args:
        model: The loaded model
        processor: The processor
        video_path: Path to video file
        prompt: User prompt
        sys_prompt: System prompt
        max_frames: Maximum frames to process

    Returns:
        Generated text response
    """
    try:
        # Load video frames
        frames = load_video_frames(video_path, max_frames=max_frames)
        print(f"Extracted {len(frames)} frames from video")

        # Create conversation with multiple images
        content = [{"type": "text", "text": f"{prompt} (Analyzing {len(frames)} frames from video)"}]

        # Add each frame as an image
        for i, frame in enumerate(frames):
            content.append({"type": "image", "image": frame})

        conversations = [
            {
                "role": "system",
                "content": [{"type": "text", "text": sys_prompt}]
            },
            {
                "role": "user",
                "content": content,
            },
        ]

        # Process the conversation
        inputs = processor.apply_chat_template(
            conversations,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )

        # Move inputs to model device
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generate response
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                pad_token_id=processor.tokenizer.eos_token_id
            )

            prompt_length = inputs["input_ids"].size(1)
            completion_ids = generated_ids[:, prompt_length:]
            text = processor.batch_decode(
                completion_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )

        return text[0] if text else "No response generated"

    except Exception as e:
        print(f"Error during multi-frame inference: {e}")
        return f"Error processing video: {str(e)}"


In [ ]:
def inference_video_native(model, processor, video_path, prompt, sys_prompt):
    """
    Native video inference using Qwen2.5-Omni's built-in video processing
    Note: This requires the full Qwen2_5OmniForConditionalGeneration model
    """
    try:
        conversations = [
            {
                "role": "system",
                "content": [{"type": "text", "text": sys_prompt}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": video_path},
                    {"type": "text", "text": prompt},
                ],
            },
        ]

        # Process the conversation with video support
        inputs = processor.apply_chat_template(
            conversations,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            video_fps=1,
            padding=True,
        )

        # Move inputs to model device
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generate response
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
            )

            # Decode the response
            text = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )

        return text[0] if text else "No response generated"

    except Exception as e:
        print(f"Error during native video inference: {e}")
        print("Note: Native video processing may require the full Qwen2_5OmniForConditionalGeneration model")
        return f"Error processing video natively: {str(e)}"


In [ ]:
# Test video loading first
video_path = "/content/frontlever.mp4"  # Update this path to your video

print("Testing video loading...")
if test_video_loading(video_path):
    print("✓ Video loading test passed!")
else:
    print("✗ Video loading test failed. Please check your video path.")
    # You can update the video_path here if needed


In [ ]:
# Configuration
prompt = "What exercise is performed in the video? Please provide detailed analysis and suggest any technique improvements."
sys_prompt = "You are a helpful fitness assistant that analyzes exercise videos and provides expert advice."

print("="*60)
print("SINGLE FRAME ANALYSIS")
print("="*60)

# Single frame inference
result_single = inference_single_frame(
    model, processor, video_path, prompt, sys_prompt, max_frames=8
)

print("\n**Single Frame Analysis Result:**")
print("-" * 40)
print(result_single)


In [ ]:
print("\n" + "="*60)
print("MULTI-FRAME ANALYSIS")
print("="*60)

# Multi-frame inference
result_multi = inference_multi_frame(
    model, processor, video_path, prompt, sys_prompt, max_frames=4
)

print("\n**Multi-Frame Analysis Result:**")
print("-" * 40)
print(result_multi)


In [ ]:
#optional
'''
print("\n" + "="*60)
print("NATIVE VIDEO PROCESSING (EXPERIMENTAL)")
print("="*60)

#Try native video processing (may not work with ThinkerForConditionalGeneration)
try:
    result_native = inference_video_native(
        model, processor, video_path, prompt, sys_prompt
    )

    print("\n**Native Video Processing Result:**")
    print("-" * 40)
    print(result_native)
except Exception as e:
    print(f"Native video processing not available with current model: {e}")
    print("This is expected when using Qwen2_5OmniThinkerForConditionalGeneration")
  '''
